In [ ]:
%pip install pandas seaborn matplotlib folium ipywidgets requests

import pandas as pd
import numpy as np
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from IPython.display import display, HTML, Markdown
import folium
from ipywidgets import interact, Dropdown, interact_manual

games = pd.read_csv("gameinfo.csv")
parks = pd.read_csv("ballparks.csv")
stadiums = pd.read_csv("stadiums.csv")

def find_lat(coord):
    coord = coord.strip("()")
    coord = coord.split(",")
    return float(coord[0])

def find_lng(coord):
    coord = coord.strip("()")
    coord = coord.split(",")
    return float(coord[1])
    
stadiums["lat"] = stadiums.apply(lambda row: find_lat(row["coordinates"]), axis = 1)
stadiums["lng"] = stadiums.apply(lambda row: find_lng(row["coordinates"]), axis = 1)

codelist = sorted(games["visteam"].unique())
teamlist = [ 'All-Star Game', 'Los Angeles Angels', 'Arizona Diamondbacks', 'Oakland Athletics', 'Atlanta Braves', 'Baltimore Orioles', 'Boston Red Sox', 'Chicago White Sox', 'Chicago Cubs', 'Cincinnati Reds', 'Cleveland Guardians', 'Colorado Rockies', 'Detroit Tigers', 'Houston Astros', 'Kansas City Royals', 'Los Angeles Dodgers', 'Miami Marlins', 'Milwaukee Brewers', 'Minnesota Twins', 'New York Yankees', 'New York Mets', 'Philadelphia Phillies', 'Pittsburgh Pirates', 'San Diego Padres', 'Seattle Mariners', 'San Francisco Giants', 'St. Louis Cardinals', 'Tampa Bay Rays', 'Texas Rangers', 'Toronto Blue Jays', 'Washington Nationals' ]

games = games.drop(['number', 'gid','innings','tiebreaker','usedh','usedh', 'htbf', "oscorer", "forfeit", "suspend", "umphome", "ump1b", "ump2b", "ump3b", "umplf", "umprf", "wp", "lp", "save", "line", "batteries", "lineups", "box", "pbp", "season"], axis=1)


def translateteamcode(code):
    for team, name in zip(codelist, teamlist):
        if team == code:
            return name

def findday(date):
    temp = datetime.strptime(str(date), "%Y%m%d")
    return temp.strftime("%A")

def get_games(team):
    temp = games[games['hometeam'] == team]
    temp1 = games[games['visteam'] == team]
    temp = pd.concat([temp, temp1])
    return temp

def get_away_games(team):
    temp = games[games['visteam'] == team]
    return temp

games['DayOfWeek'] = games.apply(lambda row: findday(row['date']), axis=1)
games['visteam'] = games.apply(lambda row: translateteamcode(row['visteam']), axis=1)
games['hometeam'] = games.apply(lambda row: translateteamcode(row['hometeam']), axis=1)
games['wteam'] = games.apply(lambda row: translateteamcode(row['wteam']), axis=1)
games['lteam'] = games.apply(lambda row: translateteamcode(row['lteam']), axis=1)


#code to run API to make stadiums df (which is loaded as a csv)
#stadiums = games[['site']].copy()
#stadiums = games[['site']].drop_duplicates().reset_index(drop=True)

#parks_lookup = parks.set_index('PARKID')
#def find_stadium(park_id):
#    row = parks_lookup.loc[park_id]
#    return f"{row['NAME']}, {row['CITY']}, {row['STATE']} "

#API_KEY = "ed911e733ac552c4ef418af7"
#BASE_URL = "https://cent.ischool-iot.net/api/google/geocode"

#headers = {
#    "accept": "application/json",
#    "X-API-KEY": API_KEY,
#}

#def find_location(stadium):
#    params = {"location": stadium}
#    response = requests.get(BASE_URL, headers=headers, params=params)
#    response.raise_for_status()
    
#    data = response.json()
#    return data["results"][0]["geometry"]["location"]["lat"], data["results"][0]["geometry"]["location"]["lng"]

#stadiums['location'] = stadiums.apply(lambda row: find_stadium(row['site']), axis=1)
#stadiums['coordinates'] = stadiums.apply(lambda row: find_location(row['location']), axis=1)
#stadiums.to_csv('stadiums.csv')

stadiums_lookup = stadiums.set_index('site')

def find_lat(park_id):
        row = stadiums_lookup.loc[park_id]
        return row['lat']

def find_lng(park_id):
    row = stadiums_lookup.loc[park_id]
    return row['lng']

def get_games(team):
    temp = games[games['hometeam'] == team]
    temp1 = games[games['visteam'] == team]
    temp = pd.concat([temp, temp1])
    return temp

def get_away_games(team):
    temp = games[games['visteam'] == team]
    return temp

games['lat'] = games.apply(lambda row: find_lat(row['site']), axis=1)
games['lng'] = games.apply(lambda row: find_lng(row['site']), axis=1)

display(HTML("<h1>2025 Baseball Record Explorer</h1>"))
display(HTML("<h2>Start by choosing a team below!</h2>"))


@interact_manual(team=teamlist)
def onclick(team):
    display(HTML(f"<h2>Let's see where the <b>{team}</b> played away from home.</h2>"))
    display(HTML(f"The bigger the circle, the more games they played there."))
    display(HTML(f"Click on a circle to see more info about the opposing team and the exact number of games."))
    
    m = folium.Map(location=[39.8283, -98.5795], zoom_start=5)

    for hometeam, group in get_away_games(team).groupby('hometeam'):
        lat = group['lat'].iloc[0]
        lng = group['lng'].iloc[0]
        
        game_count = len(group)
        
        folium.CircleMarker(
            location=[lat, lng],
            radius=game_count * 2,
            color='blue',
            fill=True,
            fillOpacity=0.6,
            popup=f"{hometeam}: {game_count} games",
        ).add_to(m)
    
    display(m)
    
    display(HTML(f"<h2>In addition to the games shown on this map, the <b>{team}</b> played <b>{len(games[games['hometeam'] == team])} </b> games at home.</h2>"))
    display(HTML(f"<h3>That's <b>{int((len(games[games['hometeam'] == team]) / len(get_games(team)))*100)}% </b> of all of their games! </h3>"))
    display(HTML("<br> </br>"))
    display(HTML(f"<h2>Lets see the rate that they win home games compared to away games!</h2>"))

    teamgames = get_games(team)
    teamgames['win'] = teamgames.apply(lambda row: True if row['wteam'] == team else False, axis=1)
    
    teamhomegames = teamgames[teamgames['hometeam'] == team]
    teamawaygames = teamgames[teamgames['visteam'] == team]
    home_win_pct = teamhomegames['win'].mean() * 100
    away_win_pct = teamawaygames['win'].mean() * 100
    
    plot_df = pd.DataFrame({
        'Location': ['Home', 'Away'],
        'Win %': [home_win_pct, away_win_pct]
    })
    
    sns.barplot(data=plot_df, x='Location', y='Win %', hue='Location')
    plt.title(f'{team}: Home vs Away Win Percentage')
    plt.ylim(0, 100)
    plt.ylabel('Win %')
    
    for i, pct in enumerate([home_win_pct, away_win_pct]):
        plt.text(i, pct + 2, f'{pct:.1f}%', ha='center')
    
    plt.show()

    overallwl = games

    overallwl['homewin'] = overallwl.apply(lambda row: True if row['wteam'] == row["hometeam"] else False, axis=1)
    overallwl['awaywin'] = overallwl.apply(lambda row: True if row['wteam'] == row["visteam"] else False, axis=1)
    
    display(HTML(f"<h2>The average home win rate across the MLB is <b>{(overallwl['homewin'].mean() * 100):.0f}%</b>. The average away win rate across the MLB is <b>{(overallwl['awaywin'].mean() * 100):.0f}%</b></h2>"))
    display(HTML("<br> </br>"))
    display(HTML(f"<h2>Let's see how the {team} do across the days of the week. </h2>"))

    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

    win_by_day = (teamgames.groupby('DayOfWeek')['win'].mean() * 100).reindex(day_order).reset_index()
    win_by_day.columns = ['DayOfWeek', 'Win %']
    
    ax = sns.barplot(data=win_by_day, x='DayOfWeek', y='Win %')
    ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3)
    
    plt.title(f'{team}: Win % by Day of Week')
    plt.ylim(0, 100)
    plt.ylabel('Win %')
    plt.xlabel('')
    plt.show()

    display(HTML("<br> </br>"))
    display(HTML(f"<h2>Let's see the regularity {team} plays with each sky condition. </h2>"))

    sky_df = teamgames['sky'].value_counts(normalize = True).reset_index()
    
    ax = sky_df.set_index('sky').plot.pie(y='proportion', autopct='%.1f%%', legend=False, figsize=(6, 6), ylabel='')
    plt.title(f"{team}: % of games played with each Sky condition")
    plt.show()
    
    display(HTML(f"<h2>Let's see the rate which {team} wins with each sky condition. </h2>"))
    
    ax = sns.barplot(data = teamgames, x = "sky", y = teamgames["win"] * 100, estimator = np.mean)
    ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3)

    plt.title(f"{team}: Win % by Sky condition")
    plt.ylim(0,100)
    plt.ylabel("Win %")
    plt.show()

    teamgames['location'] = teamgames.apply(lambda row: 'home' if row['hometeam'] == team else 'away', axis=1)

    conditions = teamgames.groupby(['location', 'DayOfWeek', 'sky']).aggregate(win_pct=('win', lambda row: row.mean() * 100), games=('win', 'size')).reset_index()
    conditions = conditions[conditions['games'] >= 3]
    
    best = conditions.sort_values('win_pct', ascending=False).iloc[0]
    
    display(HTML(f"<h2>Based off of the conditions we looked at, the {team} are most likely to win:</h2>"))
    display(HTML(f"<h3>A <b>{best['sky']} {best['location']} game on a {best['DayOfWeek']}</b>, which they won <b>{best['win_pct']:.0f}%</b> of their {int(best['games'])} games under this condition.</h3>"))